# Evaluate your AI Agent using Gen AI Evaluation Service and ADK

## Overview

This notebook guides you through evaluating an **ADK (Agent Development Kit)** agent using both **Agent Platform's Gen AI Evaluation Service** and **ADK's built-in `AgentEvaluator`**.

## Use Case: Weather Assistant

Throughout this entire notebook, we will evaluate a single, cohesive application: the **Weather Assistant** introduced in [building_agent_with_adk.ipynb](../../building_agents/solutions/building_agent_with_adk.ipynb). Our Weather Assistant has three core responsibilities:

1. **Look up city weather (`get_weather`)**: Retrieve weather reports for supported cities (`New York`, `London`, `Tokyo`) formatted in the user's preferred temperature unit (`Celsius` or `Fahrenheit`), and politely report when weather data is unavailable (e.g., `Paris`).
2. **Manage user preferences (`set_temperature_preference`)**: Save the user's preferred temperature unit (`Celsius` or `Fahrenheit`) in session state (`user:temperature_unit`) so subsequent weather lookups use that unit.
3. **Delegate to specialist sub-agents (`greeting_agent` and `farewell_agent`)**: Route greetings (`say_hello`) and farewells (`say_goodbye`) to dedicated sub-agents.

Using this single Weather Assistant across every section allows you to compare how each evaluation technique works and what failure modes it catches.

## Learning Goals

By the end of this notebook, you will understand how to:
* Set up an ADK agent with multiple tools, session state, and sub-agents for automated evaluation
* Build a comprehensive Agent Evaluation dataset with reference trajectories and ground-truth outcomes
* Evaluate **Single-Tool Usage** (`TrajectorySingleToolUse`) and **Agent Observability** (Latency and Failure Rate)
* Evaluate **Multi-Step Trajectories** (`trajectory_exact_match`, `trajectory_in_order_match`, `trajectory_any_order_match`, `trajectory_precision`, `trajectory_recall`)
* Evaluate **Final Response Quality** and **Tool-Conditioned Groundedness** using custom `PointwiseMetric` rubrics
* Evaluate **Multi-Turn Stateful Conversations** and sub-agent delegation across turns
* Diagnose trajectory vs. hallucination failures using **Bring-Your-Own-Dataset (BYOD)** evaluation
* Run **Inner-Loop Regression Tests** with ADK's built-in `AgentEvaluator` (`.test.json`) and **Native Agent Evaluation** with `vertexai.Client().evals`

For more information, see the [Gen AI Evaluation Service documentation](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/evaluation-overview) and the [ADK Evaluation documentation](https://google.github.io/adk-docs/evaluate/).

## Import libraries

In [ ]:
import asyncio
import importlib
import json
import os
import time
import uuid
import warnings
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import vertexai
from google.adk.agents import Agent
from google.adk.evaluation.agent_evaluator import AgentEvaluator
from google.adk.models.lite_llm import LiteLlm  # For multi-model support
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.tool_context import ToolContext
from google.genai import types  # For creating message Content/Parts
from IPython.display import HTML, Markdown, display

# Ignore all warnings
warnings.filterwarnings("ignore")

import logging

logging.basicConfig(level=logging.ERROR)
logging.getLogger("opentelemetry.context").setLevel(logging.CRITICAL)

In [ ]:
GEMINI_LOCATION = "global"
os.environ["GOOGLE_CLOUD_LOCATION"] = GEMINI_LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"  # Use Agent Platform API

In [ ]:
!mkdir -p adk_agents/agent1_weather_lookup

In [ ]:
%%bash
echo > adk_agents/.env "GOOGLE_CLOUD_LOCATION=$GOOGLE_CLOUD_LOCATION
GOOGLE_GENAI_USE_VERTEXAI=$GOOGLE_GENAI_USE_VERTEXAI
"

## Define helper functions

Initiate a set of plotting helper functions to visualize our evaluation results:
 - Grouped bar chart with specified evaluation metrics,
 - Displays a subset of rows including a drill-down view,
 - Plot the radar chart.

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook"


def _get_metric_name(metric) -> str:
    """Extracts the string metric name from a metric string or Metric object."""
    if isinstance(metric, str):
        return metric
    if hasattr(metric, "metric_name"):
        return str(metric.metric_name)
    return str(metric)


def plot_bar_plot(
    eval_result, title: str, metrics: list = None
) -> None:
    """Displays a grouped bar chart with specified evaluation metrics."""
    summary_metrics = eval_result.summary_metrics
    if metrics:
        normalized_metrics = [_get_metric_name(m) for m in metrics]
        summary_metrics = {
            k: v
            for k, v in summary_metrics.items()
            if any(selected_metric in k for selected_metric in normalized_metrics)
            and isinstance(v, (int, float))
        }

    fig = go.Figure(
        data=[
            go.Bar(
                x=list(summary_metrics.keys()),
                y=list(summary_metrics.values()),
                name=title,
            )
        ]
    )
    fig.update_layout(title=title, barmode="group", yaxis=dict(range=[0, 1.05]))
    fig.show()


def render_trajectory_drilldown(row: pd.Series) -> None:
    """Renders a side-by-side comparison of reference vs. predicted trajectory for a row."""
    ref_traj = row.get("reference_trajectory", "[]")
    pred_traj = row.get("predicted_trajectory", "[]")
    if isinstance(ref_traj, str):
        try:
            ref_traj = json.loads(ref_traj)
        except json.JSONDecodeError:
            pass
    if isinstance(pred_traj, str):
        try:
            pred_traj = json.loads(pred_traj)
        except json.JSONDecodeError:
            pass

    drilldown_html = f"""
    <div style="display: flex; gap: 20px; font-family: monospace; font-size: 13px; background: #f8f9fa; padding: 10px; border-radius: 6px; margin-bottom: 12px;">
        <div style="flex: 1;">
            <strong>Expected Reference Trajectory:</strong>
            <pre style="margin: 6px 0 0 0;">{json.dumps(ref_traj, indent=2)}</pre>
        </div>
        <div style="flex: 1;">
            <strong>Actual Predicted Trajectory:</strong>
            <pre style="margin: 6px 0 0 0;">{json.dumps(pred_traj, indent=2)}</pre>
        </div>
    </div>
    """
    display(HTML(drilldown_html))


def display_dataframe_rows(
    df: pd.DataFrame,
    columns: list[str] | None = None,
    num_rows: int = 3,
    display_drilldown: bool = False,
) -> None:
    """Displays a subset of rows from a DataFrame, optionally including a trajectory drill-down view."""
    if columns:
        df = df[[c for c in columns if c in df.columns]]

    base_style = "font-family: monospace; font-size: 14px; white-space: pre-wrap; width: auto; overflow-x: auto;"
    header_style = base_style + "font-weight: bold;"

    for _, row in df.head(num_rows).iterrows():
        for column in df.columns:
            display(
                HTML(
                    f"<span style='{header_style}'>{column.replace('_', ' ').title()}: </span>"
                )
            )
            display(
                HTML(f"<span style='{base_style}'>{row[column]}</span><br>")
            )

        if (
            display_drilldown
            and "predicted_trajectory" in df.columns
            and "reference_trajectory" in df.columns
        ):
            render_trajectory_drilldown(row)

        display(HTML("<hr>"))


def display_radar_plot(eval_results, title: str, metrics=None):
    """Plot a radar chart of summary metrics."""
    fig = go.Figure()
    summary_metrics = eval_results.summary_metrics
    if metrics:
        normalized_metrics = [_get_metric_name(m) for m in metrics]
        summary_metrics = {
            k: v
            for k, v in summary_metrics.items()
            if any(selected_metric in k for selected_metric in normalized_metrics)
            and isinstance(v, (int, float))
        }

    min_val = 0
    max_val = max(list(summary_metrics.values()) + [1.0])

    fig.add_trace(
        go.Scatterpolar(
            r=list(summary_metrics.values()),
            theta=list(summary_metrics.keys()),
            fill="toself",
            name=title,
        )
    )
    fig.update_layout(
        title=title,
        polar=dict(radialaxis=dict(visible=True, range=[min_val, max_val])),
        showlegend=True,
    )
    fig.show()

## Build the Weather Assistant Agent

We will build and evaluate the **Weather Assistant** agent introduced in [building_agent_with_adk.ipynb](../../building_agents/solutions/building_agent_with_adk.ipynb), combining all three core agentic capabilities in a single cohesive application:

1. **`get_weather(city, tool_context)`**: Looks up weather for `New York`, `London`, and `Tokyo`, formats the temperature according to `user:temperature_unit` (`Celsius` or `Fahrenheit`) stored in session state, saves `last_city_checked` in state, and returns a structured error for unsupported cities (such as `Paris`).
2. **`set_temperature_preference(unit, tool_context)`**: Updates `user:temperature_unit` in session state to `"Celsius"` or `"Fahrenheit"`.
3. **Sub-agent delegation (`greeting_agent` and `farewell_agent`)**: Delegates greetings (`say_hello`) and farewells (`say_goodbye`) to specialist sub-agents.

In [ ]:
%%writefile ./adk_agents/agent1_weather_lookup/tools.py
from google.adk.tools.tool_context import ToolContext


def get_weather(city: str, tool_context: ToolContext = None) -> dict:
    """Retrieves the current weather report for a specified city.

    Args:
        city (str): The name of the city (e.g., "New York", "London", "Tokyo").

    Returns:
        dict: A dictionary containing the weather information.
              Includes a 'status' key ('success' or 'error').
              If 'success', includes a 'report' key with weather details.
              If 'error', includes an 'error_message' key.
    """
    print(f"--- Tool: get_weather called for city: {city} ---")
    city_normalized = city.lower().replace(" ", "")

    preferred_unit = "Celsius"
    if tool_context is not None:
        preferred_unit = tool_context.state.get("user:temperature_unit", "Celsius")
        tool_context.state["last_city_checked"] = city

    mock_weather_db = {
        "newyork": {"temp_c": 25, "condition": "sunny", "display_name": "New York"},
        "london": {"temp_c": 15, "condition": "cloudy", "display_name": "London"},
        "tokyo": {"temp_c": 18, "condition": "experiencing light rain", "display_name": "Tokyo"},
    }

    if city_normalized in mock_weather_db:
        entry = mock_weather_db[city_normalized]
        temp_c = entry["temp_c"]
        if preferred_unit.lower() == "fahrenheit":
            temp_val = round((temp_c * 9 / 5) + 32)
            temp_str = f"{temp_val}°F"
        else:
            temp_str = f"{temp_c}°C"

        if entry["display_name"] == "Tokyo":
            report = f"Tokyo is {entry['condition']} and a temperature of {temp_str}."
        elif entry["display_name"] == "London":
            report = f"It's {entry['condition']} in London with a temperature of {temp_str}."
        else:
            report = f"The weather in {entry['display_name']} is {entry['condition']} with a temperature of {temp_str}."

        return {"status": "success", "report": report}
    else:
        return {
            "status": "error",
            "error_message": f"Sorry, I don't have weather information for '{city}'.",
        }


def set_temperature_preference(unit: str, tool_context: ToolContext = None) -> dict:
    """Sets the user's preferred temperature unit in session state.

    Args:
        unit (str): The preferred temperature unit ("Celsius" or "Fahrenheit").

    Returns:
        dict: Confirmation of the updated preference.
    """
    normalized_unit = "Fahrenheit" if unit.strip().lower().startswith("f") else "Celsius"
    print(f"--- Tool: set_temperature_preference called with unit: {normalized_unit} ---")
    if tool_context is not None:
        tool_context.state["user:temperature_unit"] = normalized_unit
    return {
        "status": "success",
        "message": f"Preferred temperature unit set to {normalized_unit}.",
    }


def say_hello(name: str = "there") -> str:
    """Provides a friendly greeting, optionally addressing the user by name.

    Args:
        name (str): The name of the person to greet. Defaults to "there".

    Returns:
        str: A friendly greeting message.
    """
    if not name or not name.strip():
        name = "there"
    print(f"--- Tool: say_hello called with name: {name} ---")
    return f"Hello, {name}!"


def say_goodbye() -> str:
    """Provides a polite farewell message to conclude the conversation."""
    print("--- Tool: say_goodbye called ---")
    return "Goodbye! Have a great day."

In [ ]:
%%writefile ./adk_agents/agent1_weather_lookup/agent.py
from google.adk.agents import Agent

from .tools import (
    get_weather,
    say_goodbye,
    say_hello,
    set_temperature_preference,
)

MODEL = "gemini-3.8-flash"

greeting_agent = Agent(
    name="greeting_agent",
    model=MODEL,
    description="Handles simple greetings and hellos using the 'say_hello' tool.",
    instruction=(
        "You are the Greeting Agent. Your ONLY task is to provide a friendly greeting "
        "using the 'say_hello' tool. If the user provides their name, pass it to the tool."
    ),
    tools=[say_hello],
)

farewell_agent = Agent(
    name="farewell_agent",
    model=MODEL,
    description="Handles simple farewells and goodbyes using the 'say_goodbye' tool.",
    instruction=(
        "You are the Farewell Agent. Your ONLY task is to provide a polite goodbye message "
        "using the 'say_goodbye' tool when the user ends the conversation."
    ),
    tools=[say_goodbye],
)

root_agent = Agent(
    name="weather_agent_v1",
    model=MODEL,
    description="Provides city weather reports, manages temperature unit preferences, and delegates greetings/farewells.",
    instruction=(
        "You are a helpful Weather Assistant.\n"
        "- When the user asks for the weather in one or more cities, call 'get_weather' for each city "
        "in the exact order the cities are mentioned by the user.\n"
        "- If the user asks to change or set their preferred temperature unit (Celsius or Fahrenheit), "
        "call 'set_temperature_preference' BEFORE calling 'get_weather'.\n"
        "- If 'get_weather' returns an error for a city, politely inform the user that weather data "
        "is unavailable for that city and never invent or guess weather conditions.\n"
        "- If 'get_weather' succeeds, clearly state the weather report returned by the tool.\n"
        "- Delegate standalone greetings to 'greeting_agent' and standalone farewells to 'farewell_agent'."
    ),
    tools=[get_weather, set_temperature_preference],
    sub_agents=[greeting_agent, farewell_agent],
)

In [ ]:
with open("./adk_agents/agent1_weather_lookup/__init__.py", "w") as f:
    f.write("from . import agent\n")

from adk_agents.agent1_weather_lookup import agent, tools

importlib.reload(tools)
importlib.reload(agent)

# Quick sanity check of the tools
print(agent.get_weather("New York"))
print(agent.get_weather("Paris"))
print(agent.set_temperature_preference("Fahrenheit"))

### Setup Runner and Session Service

To manage conversations and execute the agent, we need two more components:

* `SessionService`: Responsible for managing conversation history and state for different users and sessions. The `InMemorySessionService` is a simple implementation that stores everything in memory, suitable for testing and simple applications. It keeps track of the messages exchanged. We'll explore state persistence more in Step 4\.  
* `Runner`: The engine that orchestrates the interaction flow. It takes user input, routes it to the appropriate agent, manages calls to the LLM and tools based on the agent's logic, handles session updates via the `SessionService`, and yields events representing the progress of the interaction.

Let's define some constants first.

In [ ]:
APP_NAME = "weather_info_app"
USER_ID = "user_1"
SESSION_ID = "session_001"

#### Define a function for Session and Runner

In [ ]:
async def setup_session_and_runner(
    session_id: str | None = None,
    initial_state: dict | None = None,
):
    session_service = InMemorySessionService()
    resolved_session_id = session_id or f"session_{uuid.uuid4().hex[:8]}"
    example_session = await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=resolved_session_id,
        state=initial_state or {},
    )

    runner = Runner(
        agent=agent.root_agent,
        app_name=APP_NAME,
        session_service=session_service,
    )
    return example_session, runner, session_service

### Interact with the Agent and Capture Full Execution Traces

To evaluate an agent thoroughly, we need more than just its final text reply—we need the **chronological trajectory of tool calls**, the **outputs returned by those tools**, and any **updates to session state**.

We'll define an `async` helper function (`call_agent_async`) that:

1. Packages the user query into ADK's `types.Content` format.
2. Runs the agent via `runner.run_async`.
3. Collects every `function_call` in chronological order (without deduplicating legitimate repeated calls) into `predicted_trajectory`.
4. Collects every `function_response` into `tool_outputs` so we can verify whether the agent's final answer is grounded in what the tools actually returned.
5. Captures the final response and updated `session.state`.

### Run the Conversation

Finally, let's test our setup by sending a few queries to the agent. We wrap our `async` calls in a main `async` function and run it using `await`.

Watch the output:

* See the user queries.  
* Notice the `--- Tool: get_weather called... ---` logs when the agent uses the tool.  
* Observe the agent's final responses, including how it handles the case where weather data isn't available (for Paris).

In [ ]:
async def call_agent_async(
    query: str,
    runner: Runner | None = None,
    session_service: InMemorySessionService | None = None,
    session_id: str | None = None,
    include_transfers: bool = False,
) -> dict:
    """Executes a single user query against the Weather Assistant and extracts trajectory, tool outputs, and state."""
    print(f"\n>>> User Query: {query}")

    content = types.Content(role="user", parts=[types.Part(text=query)])
    if runner is None or session_service is None or session_id is None:
        session, runner, session_service = await setup_session_and_runner(
            session_id=session_id
        )
        session_id = session.id

    events = runner.run_async(
        user_id=USER_ID, session_id=session_id, new_message=content
    )

    final_response = ""
    predicted_trajectory_list = []
    tool_outputs_list = []

    async for event in events:
        if not event.content or not event.content.parts:
            continue

        for part in event.content.parts:
            if part.function_call:
                if not include_transfers and part.function_call.name == "transfer_to_agent":
                    continue
                tool_info = {
                    "tool_name": part.function_call.name,
                    "tool_input": dict(part.function_call.args),
                }
                predicted_trajectory_list.append(tool_info)

            if part.function_response:
                if not include_transfers and part.function_response.name == "transfer_to_agent":
                    continue
                tool_outputs_list.append(
                    {
                        "tool_name": part.function_response.name,
                        "tool_output": dict(part.function_response.response)
                        if part.function_response.response
                        else {},
                    }
                )

            if event.content.role == "model" and part.text:
                final_response = part.text.strip()

        if event.is_final_response() and event.content.parts[0].text:
            final_response = event.content.parts[0].text.strip()
            print("Agent Response:", final_response)

    updated_session = await session_service.get_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=session_id
    )
    current_state = dict(updated_session.state) if updated_session else {}

    return {
        "response": str(final_response),
        "predicted_trajectory": json.dumps(predicted_trajectory_list),
        "tool_outputs": json.dumps(tool_outputs_list),
        "session_state": current_state,
    }

## Evaluating the Weather Assistant with Gen AI Evaluation

When evaluating AI agents, we assess two complementary dimensions: **task quality (monitoring)** and **system health (observability)**.

**Task Quality (Monitoring)** focuses on how well our Weather Assistant reasons and acts:
* **Single Tool Selection**: Does the Weather Assistant invoke `get_weather` when asked for a forecast?
* **Multi-Step Trajectory**: Does it call tools in the right sequence—for example, calling `get_weather` for multiple cities in the requested order, or calling `set_temperature_preference` *before* `get_weather` when a user switches to Fahrenheit?
* **Response Generation & Groundedness**: Does the final answer accurately report the temperatures returned by `get_weather`, and politely state when a city (like Paris) is unavailable rather than hallucinating a forecast?

**System Health (Observability)** tracks operational reliability during inference:
* **Latency (`latency_in_seconds`)**: How long does each agent invocation take end-to-end?
* **Failure Rate (`failure`)**: How often does an agent invocation raise an unhandled error or fail to return a response?

The Gen AI Evaluation Service computes both task quality metrics and observability metrics (`latency_in_seconds` and `failure`) automatically when evaluating a runnable agent. See [Define your evaluation metrics](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/determine-eval) for details.

### Prepare the Weather Assistant Evaluation Dataset

To evaluate our Weather Assistant across all techniques in this notebook, we define a shared evaluation dataset containing:

* `prompt`: Representative user requests to the Weather Assistant (single-city queries, unsupported-city queries, multi-city queries, and combined unit-preference + weather queries).
* `reference_trajectory`: The expected chronological sequence of tool calls and arguments (`tool_name` and `tool_input`).
* `reference`: The expected ground-truth response based on the mock weather database and tool outputs.

In [ ]:
eval_data = {
    "prompt": [
        "Tell me the weather in New York",
        "What is the weather like in London?",
        "How about Paris?",
        "Tell me the weather in New York and London?",
        "Set my preferred temperature unit to Fahrenheit and tell me the weather in Tokyo.",
        "Tell me the weather in New York, London, and Paris?",
    ],
    "reference_trajectory": [
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "New York"},
            }
        ],
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "London"},
            }
        ],
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "Paris"},
            }
        ],
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "New York"},
            },
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "London"},
            },
        ],
        [
            {
                "tool_name": "set_temperature_preference",
                "tool_input": {"unit": "Fahrenheit"},
            },
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "Tokyo"},
            },
        ],
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "New York"},
            },
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "London"},
            },
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "Paris"},
            },
        ],
    ],
    "reference": [
        "The weather in New York is sunny with a temperature of 25°C.",
        "It's cloudy in London with a temperature of 15°C.",
        "Sorry, I don't have weather information for Paris.",
        "The weather in New York is sunny with a temperature of 25°C, and in London it is cloudy with a temperature of 15°C.",
        "Your preferred temperature unit has been set to Fahrenheit. Tokyo is experiencing light rain with a temperature of 64°F.",
        "The weather in New York is sunny with a temperature of 25°C, and London is cloudy with a temperature of 15°C. However, I don't have weather information for Paris.",
    ],
}

eval_sample_dataset = pd.DataFrame(eval_data)

Print some samples from the dataset.

In [ ]:
display(eval_sample_dataset)

In [ ]:
display_dataframe_rows(eval_sample_dataset, num_rows=3)

### Single tool usage evaluation

After you've set your AI agent and the evaluation dataset, you start evaluating if the agent is choosing the correct single tool for a given task.

#### Set single tool usage metrics

The `TrajectorySingleToolUse` metric checks whether the agent called a specific expected tool at least once during its trajectory, regardless of how many other tools were called or in what order.

For our Weather Assistant, every prompt in `eval_sample_dataset` asks for a city forecast, so we expect the `get_weather` tool to be invoked in every trajectory.

In [ ]:
from vertexai.preview.evaluation import EvalTask
from vertexai.preview.evaluation.metrics import (
    PointwiseMetric,
    PointwiseMetricPromptTemplate,
    TrajectorySingleToolUse,
)

In [ ]:
single_tool_usage_metrics = [TrajectorySingleToolUse(tool_name="get_weather")]

#### Run an evaluation task

To run the evaluation, you initiate an `EvalTask` using the pre-defined dataset (`eval_sample_dataset`) and metrics (`single_tool_usage_metrics` in this case) within an experiment. Then, you run the evaluation using agent_parsed_outcome function and assigns a unique identifier to this specific evaluation run, storing and visualizing the evaluation results.

In [ ]:
EXPERIMENT_NAME = "evaluate-adk-agent-v1"
PROJECT = !gcloud config get project
PROJECT = PROJECT[0]
BUCKET_NAME = f"agent-evaluation-{PROJECT}-bucket"
BUCKET_URI = f"gs://{BUCKET_NAME}"

**Checking for the existence of BUCKET. Creating it if it doesn't exist:**

In [ ]:
!gcloud storage ls --buckets | grep -w $BUCKET_URI || gcloud storage buckets create --location=us-central1 $BUCKET_URI

In [ ]:
import random
import string
import uuid
from typing import Any


def get_id() -> str:
    """Generate a uuid"""
    return str(uuid.uuid4())

Here we wrap the `call_agent_async` in a synchronous function so that we can pass it to the evaluation service.

In [ ]:
def agent_parsed_outcome(query: str) -> dict:
    """Synchronous wrapper around call_agent_async for EvalTask.evaluate(runnable=...)."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(call_agent_async(query))

    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(lambda: asyncio.run(call_agent_async(query))).result()

In [ ]:
EXPERIMENT_RUN = f"single-metric-eval-{get_id()}"

single_tool_call_eval_task = EvalTask(
    dataset=eval_sample_dataset,
    metrics=single_tool_usage_metrics,
    experiment=EXPERIMENT_NAME,
    output_uri_prefix=BUCKET_URI + "/single-metric-eval",
)

single_tool_call_eval_result = single_tool_call_eval_task.evaluate(
    runnable=agent_parsed_outcome, experiment_run_name=EXPERIMENT_RUN
)

print(single_tool_call_eval_result)

In [ ]:
def display_eval_report(eval_result) -> None:
    """Display summary metrics (including observability metrics) and row-level results."""
    metrics_df = pd.DataFrame.from_dict(
        eval_result.summary_metrics, orient="index"
    ).T
    display(Markdown("### Summary Metrics (Task Quality & Observability)"))
    display(metrics_df)

    display(Markdown("### Row-wise Metrics"))
    display(eval_result.metrics_table)

In [ ]:
display_eval_report(single_tool_call_eval_result)

# Inspect the automatically captured Observability metrics (Latency & Failure Rate)
observability_cols = [
    c
    for c in ["prompt", "latency_in_seconds", "failure", "trajectory_single_tool_use/score"]
    if c in single_tool_call_eval_result.metrics_table.columns
]
display(Markdown("### Agent Observability Breakdown (Latency & Failure Rate)"))
display(single_tool_call_eval_result.metrics_table[observability_cols])

### Trajectory Evaluation

After evaluating the agent's ability to select the single most appropriate tool for a given task, you generalize the evaluation by analyzing the tool sequence choices with respect to the user input (trajectory). This assesses whether the agent not only chooses the right tools but also utilizes them in a rational and effective order.

#### Set trajectory metrics

To evaluate agent's trajectory, Gen AI Evaluation provides several ground-truth based metrics:

* `trajectory_exact_match`: identical trajectories (same actions, same order)

* `trajectory_in_order_match`: reference actions present in predicted trajectory, in order (extras allowed)

* `trajectory_any_order_match`: all reference actions present in predicted trajectory (order, extras don't matter).

* `trajectory_precision`: proportion of predicted actions present in reference

* `trajectory_recall`: proportion of reference actions present in predicted.  

All metrics score 0 or 1, except `trajectory_precision` and `trajectory_recall` which range from 0 to 1.

In [ ]:
trajectory_metrics = [
    "trajectory_exact_match",
    "trajectory_in_order_match",
    "trajectory_any_order_match",
    "trajectory_precision",
    "trajectory_recall",
]

#### Run an evaluation task

Submit an evaluation by running `evaluate` method of the new `EvalTask`.

In [ ]:
EXPERIMENT_RUN = f"trajectory-{get_id()}"

trajectory_eval_task = EvalTask(
    dataset=eval_sample_dataset,
    metrics=trajectory_metrics,
    experiment=EXPERIMENT_NAME,
    output_uri_prefix=BUCKET_URI + "/multiple-metric-eval",
)

trajectory_eval_result = trajectory_eval_task.evaluate(
    runnable=agent_parsed_outcome, experiment_run_name=EXPERIMENT_RUN
)

In [ ]:
display_eval_report(trajectory_eval_result)

#### Visualize evaluation results

Print and visualize a sample of evaluation results.

In [ ]:
display_dataframe_rows(
    trajectory_eval_result.metrics_table,
    num_rows=4,
    display_drilldown=True,
)

In [ ]:
plot_bar_plot(
    trajectory_eval_result,
    title="Trajectory Metrics",
    metrics=[f"{metric}/mean" for metric in trajectory_metrics],
)

### Evaluate final response

Similar to model evaluation, you can evaluate the final response of the agent using Gen AI Evaluation.

#### Set response metrics

After agent inference, Gen AI Evaluation provides several metrics to evaluate generated responses. You can use computation-based metrics to compare the response to a reference (if needed) and using existing or custom model-based metrics to determine the quality of the final response.

Check out the [documentation](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/determine-eval) to learn more.


In [ ]:
response_metrics = ["safety", "coherence", "rouge_l_sum"]

#### Run an evaluation task

To evaluate agent's generated responses, use the `evaluate` method of the EvalTask class.

In [ ]:
EXPERIMENT_RUN = f"response-{get_id()}"

response_eval_task = EvalTask(
    dataset=eval_sample_dataset,
    metrics=response_metrics,
    experiment=EXPERIMENT_NAME,
    output_uri_prefix=BUCKET_URI + "/response-metric-eval",
)

response_eval_result = response_eval_task.evaluate(
    runnable=agent_parsed_outcome, experiment_run_name=EXPERIMENT_RUN
)

display_eval_report(response_eval_result)

#### Visualize evaluation results


Print new evaluation result sample.

In [ ]:
display_dataframe_rows(response_eval_result.metrics_table, num_rows=3)

### Evaluate Tool-Conditioned Response Groundedness

Standard text metrics like `coherence` only evaluate whether the response reads well—not whether the Weather Assistant stayed faithful to its tools. For example, if `get_weather("Paris")` returns an error, a fluent but hallucinated forecast (`"It is sunny and 22°C in Paris"`) would score high on `coherence` while failing the user!

To catch tool hallucinations and ungrounded answers, we define a custom LLM-as-a-judge metric (`response_follows_trajectory`) that inspects:
1. `prompt`: What the user asked the Weather Assistant.
2. `predicted_trajectory`: Which tools and arguments the Weather Assistant called.
3. `reference`: The expected ground-truth outcome from the tools (e.g., `25°C` for New York, `64°F` for Tokyo in Fahrenheit, or an unavailable-city notice for Paris).
4. `response` (automatically included by `PointwiseMetricPromptTemplate`): The Weather Assistant's final answer.

#### Define a custom metric

According to the [documentation](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/determine-eval#model-based-metrics), you can define a prompt template for evaluating whether an AI agent's response follows logically from its actions by setting up criteria and a rating system for this evaluation.

Define a `criteria` to set the evaluation guidelines and a `pointwise_rating_rubric` to provide a scoring system (1 or 0). Then use a `PointwiseMetricPromptTemplate` to create the template using these components.


In [ ]:
criteria = {
    "Follows trajectory and tool outcomes": (
        "Evaluate whether the Weather Assistant's final response logically follows from "
        "the sequence of tool calls it took (`predicted_trajectory`) and accurately reflects "
        "the expected ground-truth tool outcome (`reference`). Consider these sub-points:\n"
        "  - Does the response report the exact temperature, unit (°C or °F), and weather condition from the tool?\n"
        "  - If a city is unsupported (such as Paris), does the response politely state that weather information "
        "is unavailable without hallucinating a forecast?\n"
        "  - Are there any contradictions or fabricated details not supported by the trajectory and reference?"
    )
}

pointwise_rating_rubric = {
    "1": "Follows trajectory and accurately reflects tool outcomes without hallucination",
    "0": "Does not follow trajectory, omits requested cities, or hallucinates weather data",
}

response_follows_trajectory_prompt_template = PointwiseMetricPromptTemplate(
    criteria=criteria,
    rating_rubric=pointwise_rating_rubric,
    input_variables=["prompt", "predicted_trajectory", "reference"],
)

Print the prompt_data of this template containing the combined criteria and rubric information ready for use in an evaluation.

In [ ]:
print(response_follows_trajectory_prompt_template.prompt_data)

After you define the evaluation prompt template, set up the associated metric to evaluate how well a response follows a specific trajectory. The `PointwiseMetric` creates a metric where `response_follows_trajectory` is the metric's name and `response_follows_trajectory_prompt_template` provides instructions or context for evaluation you set up before.

In [ ]:
response_follows_trajectory_metric = PointwiseMetric(
    metric="response_follows_trajectory",
    metric_prompt_template=response_follows_trajectory_prompt_template,
)

#### Set response metrics

Set new generated response evaluation metrics by including the custom metric.


In [ ]:
response_tool_metrics = [
    "trajectory_exact_match",
    "trajectory_in_order_match",
    "safety",
    response_follows_trajectory_metric,
]

#### Run an evaluation task

Run a new agent's evaluation.

In [ ]:
EXPERIMENT_RUN = f"response-over-tools-{get_id()}"

response_eval_tool_task = EvalTask(
    dataset=eval_sample_dataset,
    metrics=response_tool_metrics,
    experiment=EXPERIMENT_NAME,
    output_uri_prefix=BUCKET_URI + "/reasoning-metric-eval",
)

response_eval_tool_result = response_eval_tool_task.evaluate(
    runnable=agent_parsed_outcome, experiment_run_name=EXPERIMENT_RUN
)

display_eval_report(response_eval_tool_result)

#### Visualize evaluation results

Visualize evaluation result sample.

In [ ]:
display_dataframe_rows(
    response_eval_tool_result.metrics_table,
    num_rows=4,
    display_drilldown=True,
)

In [ ]:
plot_bar_plot(
    response_eval_tool_result,
    title="Trajectory & Grounded Response Metrics",
    metrics=[f"{_get_metric_name(metric)}/mean" for metric in response_tool_metrics],
)

## Bring-Your-Own-Dataset (BYOD): Diagnosing Trajectory vs. Grounding Failures

In [Bring-Your-Own-Dataset (BYOD)](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/evaluation-dataset) scenarios, you evaluate pre-recorded agent logs (`predicted_trajectory` and `response`) without re-running agent inference.

BYOD is especially useful for regression-testing your evaluation metrics and diagnosing **why** an agent failed. Below, we test four representative Weather Assistant scenarios:
1. **Pass (New York)**: Correct tool call and accurate `25°C` report.
2. **Wrong City Response (London)**: Agent calls `get_weather("London")` (`trajectory_exact_match = 1.0`), but mistakenly reports New York's weather (`response_follows_trajectory = 0.0`).
3. **Hallucinated Forecast on Tool Error (Paris)**: Agent calls `get_weather("Paris")` (`trajectory_exact_match = 1.0`), which returns an error, but the agent fabricates a sunny `22°C` forecast (`response_follows_trajectory = 0.0`).
4. **Reversed Tool Order (Tokyo in Fahrenheit)**: Agent calls `get_weather("Tokyo")` *before* `set_temperature_preference("Fahrenheit")` (`trajectory_exact_match = 0.0`), resulting in a Celsius report instead of Fahrenheit.

### Prepare the BYOD Evaluation Dataset

Notice that we include `prompt`, `reference_trajectory`, `predicted_trajectory`, `reference`, and `response` for our four diagnostic Weather Assistant cases.

In [ ]:
byod_eval_data = {
    "prompt": [
        "Tell me the weather in New York",
        "What is the weather like in London?",
        "How about Paris?",
        "Set my preferred temperature unit to Fahrenheit and tell me the weather in Tokyo.",
    ],
    "reference_trajectory": [
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "New York"},
            }
        ],
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "London"},
            }
        ],
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "Paris"},
            }
        ],
        [
            {
                "tool_name": "set_temperature_preference",
                "tool_input": {"unit": "Fahrenheit"},
            },
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "Tokyo"},
            },
        ],
    ],
    "predicted_trajectory": [
        # 1. Exact match + grounded response
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "New York"},
            }
        ],
        # 2. Exact trajectory match, but wrong city in response
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "London"},
            }
        ],
        # 3. Exact trajectory match, but hallucinated weather when tool returned error
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "Paris"},
            }
        ],
        # 4. Out-of-order tool execution (get_weather called before set_temperature_preference)
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "Tokyo"},
            },
            {
                "tool_name": "set_temperature_preference",
                "tool_input": {"unit": "Fahrenheit"},
            },
        ],
    ],
    "reference": [
        "The weather in New York is sunny with a temperature of 25°C.",
        "It's cloudy in London with a temperature of 15°C.",
        "Sorry, I don't have weather information for Paris.",
        "Your preferred temperature unit has been set to Fahrenheit. Tokyo is experiencing light rain with a temperature of 64°F.",
    ],
    "response": [
        "The weather in New York is sunny with a temperature of 25°C.",
        "The weather in New York is sunny with a temperature of 25°C.",
        "The weather in Paris is sunny and warm with a temperature of 22°C.",
        "Tokyo is experiencing light rain and a temperature of 18°C. I have also updated your unit preference to Fahrenheit.",
    ],
}

byod_eval_sample_dataset = pd.DataFrame(byod_eval_data)
byod_eval_sample_dataset["predicted_trajectory"] = byod_eval_sample_dataset[
    "predicted_trajectory"
].apply(json.dumps)
byod_eval_sample_dataset["reference_trajectory"] = byod_eval_sample_dataset[
    "reference_trajectory"
].apply(json.dumps)

### Run an evaluation task

Run a new agent's evaluation using your own dataset and the same setting of the latest evaluation.

In [ ]:
EXPERIMENT_RUN_NAME = f"response-over-tools-byod-{get_id()}"

byod_response_eval_tool_task = EvalTask(
    dataset=byod_eval_sample_dataset,
    metrics=response_tool_metrics,
    experiment=EXPERIMENT_NAME,
    output_uri_prefix=BUCKET_URI + "/byod-eval",
)

byod_response_eval_tool_result = byod_response_eval_tool_task.evaluate(
    experiment_run_name=EXPERIMENT_RUN_NAME
)

display_eval_report(byod_response_eval_tool_result)

### Visualize evaluation results

Visualize evaluation result sample.

In [ ]:
display_dataframe_rows(
    byod_response_eval_tool_result.metrics_table,
    num_rows=4,
    display_drilldown=True,
)

In [ ]:
display_radar_plot(
    byod_response_eval_tool_result,
    title="Weather Assistant BYOD Evaluation",
    metrics=[f"{_get_metric_name(metric)}/mean" for metric in response_tool_metrics],
)

## Evaluating Multi-Turn Stateful Conversations and Sub-Agent Delegation

Real-world agents rarely operate in isolated single turns. Our Weather Assistant maintains **session state** (`user:temperature_unit` and `last_city_checked`) across turns and delegates greetings and farewells to specialist **sub-agents** (`greeting_agent` and `farewell_agent`).

Let's evaluate a 4-turn conversation within a single persistent session:
1. **Turn 1 (Sub-Agent Delegation)**: `"Hello, my name is Alex!"` $\rightarrow$ Root agent transfers to `greeting_agent`, which calls `say_hello(name="Alex")`.
2. **Turn 2 (State Mutation)**: `"Please set my preferred temperature unit to Fahrenheit."` $\rightarrow$ Calls `set_temperature_preference(unit="Fahrenheit")` and updates `session.state["user:temperature_unit"]`.
3. **Turn 3 (State-Dependent Tool Execution)**: `"What is the weather in New York?"` $\rightarrow$ Calls `get_weather(city="New York")`, reads `"Fahrenheit"` from `session.state`, and reports `77°F`.
4. **Turn 4 (Sub-Agent Delegation)**: `"Thanks for your help, goodbye!"` $\rightarrow$ Transfers to `farewell_agent`, which calls `say_goodbye()`.

In [ ]:
async def run_multi_turn_weather_conversation():
    """Runs a 4-turn stateful conversation with the Weather Assistant and records per-turn traces and state."""
    session, runner, session_service = await setup_session_and_runner(
        session_id=f"multi_turn_{get_id()}"
    )

    turns = [
        {
            "prompt": "Hello, my name is Alex!",
            "reference_trajectory": [
                {"tool_name": "transfer_to_agent", "tool_input": {"agent_name": "greeting_agent"}},
                {"tool_name": "say_hello", "tool_input": {"name": "Alex"}},
            ],
            "reference": "Hello, Alex!",
        },
        {
            "prompt": "Please set my preferred temperature unit to Fahrenheit.",
            "reference_trajectory": [
                {"tool_name": "set_temperature_preference", "tool_input": {"unit": "Fahrenheit"}},
            ],
            "reference": "Preferred temperature unit set to Fahrenheit.",
        },
        {
            "prompt": "What is the weather in New York?",
            "reference_trajectory": [
                {"tool_name": "get_weather", "tool_input": {"city": "New York"}},
            ],
            "reference": "The weather in New York is sunny with a temperature of 77°F.",
        },
        {
            "prompt": "Thanks for your help, goodbye!",
            "reference_trajectory": [
                {"tool_name": "transfer_to_agent", "tool_input": {"agent_name": "farewell_agent"}},
                {"tool_name": "say_goodbye", "tool_input": {}},
            ],
            "reference": "Goodbye! Have a great day.",
        },
    ]

    records = []
    for turn_idx, turn in enumerate(turns, start=1):
        outcome = await call_agent_async(
            query=turn["prompt"],
            runner=runner,
            session_service=session_service,
            session_id=session.id,
            include_transfers=True,
        )
        records.append(
            {
                "turn": turn_idx,
                "prompt": turn["prompt"],
                "response": outcome["response"],
                "predicted_trajectory": outcome["predicted_trajectory"],
                "reference_trajectory": json.dumps(turn["reference_trajectory"]),
                "reference": turn["reference"],
                "session_state": outcome["session_state"],
            }
        )

    return pd.DataFrame(records)


multi_turn_df = await run_multi_turn_weather_conversation()
display(multi_turn_df[["turn", "prompt", "response", "predicted_trajectory", "session_state"]])

In [ ]:
# Verify final session state assertions and evaluate the multi-turn trajectories with EvalTask
final_state = multi_turn_df.iloc[-1]["session_state"]
print("Final Session State:", final_state)
assert final_state.get("user:temperature_unit") == "Fahrenheit", "Expected Fahrenheit preference in session state!"
assert final_state.get("last_city_checked") == "New York", "Expected New York in last_city_checked state!"

multi_turn_eval_task = EvalTask(
    dataset=multi_turn_df[["prompt", "response", "predicted_trajectory", "reference_trajectory", "reference"]],
    metrics=[
        "trajectory_exact_match",
        "trajectory_in_order_match",
        response_follows_trajectory_metric,
    ],
    experiment=EXPERIMENT_NAME,
    output_uri_prefix=BUCKET_URI + "/multi-turn-eval",
)

multi_turn_eval_result = multi_turn_eval_task.evaluate(
    experiment_run_name=f"multi-turn-weather-{get_id()}"
)

display_eval_report(multi_turn_eval_result)

## Inner-Loop Evaluation with ADK's Built-in `AgentEvaluator`

While Agent Platform's Gen AI Evaluation Service is ideal for large-scale experiment tracking and custom LLM-as-a-judge rubrics (**outer loop**), ADK also includes a built-in **`AgentEvaluator`** for rapid local unit/regression testing and CI/CD pipelines (`pytest` or `adk eval` CLI) (**inner loop**).

In ADK, you define golden evaluation cases in a `.test.json` (or `.evalset.json`) file backed by ADK's `EvalSet` schema, along with an optional `test_config.json` that sets pass/fail thresholds for `tool_trajectory_avg_score` and `response_match_score`. You can also export these files directly from interactive sessions in the `adk web` UI.

Let's create a `test_config.json` and `weather_agent.test.json` for our **Weather Assistant** and run `AgentEvaluator.evaluate()`.

In [ ]:
%%writefile ./adk_agents/agent1_weather_lookup/test_config.json
{
  "criteria": {
    "tool_trajectory_avg_score": {
      "threshold": 1.0,
      "match_type": "EXACT"
    },
    "response_match_score": 0.4
  }
}

In [ ]:
%%writefile ./adk_agents/agent1_weather_lookup/weather_agent.test.json
{
  "eval_set_id": "weather_assistant_unit_tests",
  "name": "Weather Assistant Regression Suite",
  "description": "Validates single-city lookup, preference updates, and unsupported-city handling for the Weather Assistant.",
  "eval_cases": [
    {
      "eval_id": "new_york_weather_celsius",
      "conversation": [
        {
          "invocation_id": "inv-ny-001",
          "user_content": {
            "parts": [{"text": "Tell me the weather in New York"}],
            "role": "user"
          },
          "final_response": {
            "parts": [{"text": "The weather in New York is sunny with a temperature of 25°C."}],
            "role": "model"
          },
          "intermediate_data": {
            "tool_uses": [
              {
                "name": "get_weather",
                "args": {"city": "New York"}
              }
            ],
            "intermediate_responses": []
          }
        }
      ],
      "session_input": {
        "app_name": "weather_info_app",
        "user_id": "user_1",
        "state": {}
      }
    },
    {
      "eval_id": "tokyo_weather_fahrenheit",
      "conversation": [
        {
          "invocation_id": "inv-tokyo-001",
          "user_content": {
            "parts": [{"text": "Set my preferred temperature unit to Fahrenheit and tell me the weather in Tokyo."}],
            "role": "user"
          },
          "final_response": {
            "parts": [{"text": "Your preferred temperature unit is set to Fahrenheit. Tokyo is experiencing light rain with a temperature of 64°F."}],
            "role": "model"
          },
          "intermediate_data": {
            "tool_uses": [
              {
                "name": "set_temperature_preference",
                "args": {"unit": "Fahrenheit"}
              },
              {
                "name": "get_weather",
                "args": {"city": "Tokyo"}
              }
            ],
            "intermediate_responses": []
          }
        }
      ],
      "session_input": {
        "app_name": "weather_info_app",
        "user_id": "user_1",
        "state": {}
      }
    }
  ]
}

In [ ]:
# Run ADK's built-in AgentEvaluator against our Weather Assistant test suite
adk_eval_csv_path = "./adk_agents/agent1_weather_lookup/adk_eval_results.csv"

await AgentEvaluator.evaluate(
    agent_module="adk_agents.agent1_weather_lookup.agent",
    eval_dataset_file_path_or_dir="./adk_agents/agent1_weather_lookup/weather_agent.test.json",
    num_runs=1,
    print_detailed_results=True,
    output_file=adk_eval_csv_path,
)

if os.path.exists(adk_eval_csv_path):
    display(pd.read_csv(adk_eval_csv_path))

## Native Agent Evaluation with `vertexai.Client().evals`

In addition to `EvalTask`, the Agent Platform SDK provides a streamlined `client.evals` interface (`vertexai.Client`) that accepts your ADK `Agent` object directly in `client.evals.run_inference(agent=..., src=...)`. It automatically executes the agent, captures all intermediate tool calls and responses (`intermediate_events`), and evaluates built-in agent rubric metrics such as `TOOL_USE_QUALITY`, `FINAL_RESPONSE_QUALITY`, `HALLUCINATION`, and `SAFETY` without requiring a custom event-parsing function.

In [ ]:
from vertexai import types as vertexai_types

eval_client = vertexai.Client(project=PROJECT, location="us-central1")

# Run inference directly against the local ADK Weather Assistant
agent_inference_dataset = eval_client.evals.run_inference(
    agent=agent.root_agent,
    src=eval_sample_dataset[["prompt", "reference"]],
)

# Evaluate using prebuilt rubric-based agent metrics
client_eval_result = eval_client.evals.evaluate(
    dataset=agent_inference_dataset,
    metrics=[
        vertexai_types.RubricMetric.TOOL_USE_QUALITY,
        vertexai_types.RubricMetric.FINAL_RESPONSE_QUALITY,
        vertexai_types.RubricMetric.HALLUCINATION,
        vertexai_types.RubricMetric.SAFETY,
    ],
)

display(client_eval_result.summary_metrics)

## Summary and Key Takeaways

In this notebook, we evaluated a single **Weather Assistant** agent across every layer of modern agent evaluation:

| Evaluation Approach | What It Measures on the Weather Assistant | Key Metrics / Classes | Best Used For |
| --- | --- | --- | --- |
| **Single-Tool & Observability (`EvalTask`)** | Did the agent call `get_weather`, how long did it take, and did any calls fail? | `TrajectorySingleToolUse`, `latency_in_seconds`, `failure` | Verifying core tool invocation and runtime health |
| **Multi-Step Trajectory (`EvalTask`)** | Did the agent call tools in the right sequence (e.g., `New York` then `London`, or `set_temperature_preference` before `get_weather`)? | `trajectory_exact_match`, `trajectory_in_order_match`, `trajectory_any_order_match`, `trajectory_precision`, `trajectory_recall` | Validating multi-tool ordering and multi-step planning |
| **Tool-Conditioned Groundedness (`EvalTask`)** | Did the final answer report the exact tool output (`25°C` / `64°F`) and avoid hallucinating forecasts when a city (`Paris`) returned an error? | `safety`, `coherence`, `rouge_l_sum`, custom `PointwiseMetric` (`response_follows_trajectory`) | Catching post-tool hallucinations and ungrounded responses |
| **Multi-Turn & Stateful Evaluation** | Did the agent delegate to `greeting_agent`/`farewell_agent` and persist `user:temperature_unit` across turns? | Multi-turn `EvalTask` + `session.state` assertions | Testing conversational memory and multi-agent routing |
| **Bring-Your-Own-Dataset (BYOD)** | How do metrics behave on pre-recorded logs with known trajectory or grounding bugs? | `EvalTask` without `runnable` | Offline log analysis and metric calibration |
| **ADK `AgentEvaluator` & `client.evals`** | Fast local regression testing (`.test.json`) and zero-boilerplate agent evaluation | `AgentEvaluator.evaluate()`, `client.evals.run_inference()`, `types.RubricMetric` | CI/CD unit tests (`pytest` / `adk eval`) and managed rubric evaluation |

Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at

https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.